In [5]:
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import linregress

#### Reuse Codes Copied from RandomForest.ipynb

In [6]:
class Node:
    def __init__(self, data: pd.DataFrame = None, labels=None, min_samples_leaf: int = 1, max_features: float = 0.5):
        self.data = data
        self.labels = np.asarray(labels) if labels is not None else None
        self.left = None
        self.right = None
        self.split = None
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.isLeaf = (self.data is None) or (len(self.data) < 2 * self.min_samples_leaf)

    def gini_index(self, labels):
        C = Counter(labels)
        total = len(labels)
        frequencies = {k: v / total for (k, v) in C.items()}
        return float(np.sum([p * (1 - p) for p in frequencies.values()]))

    def find_best_split(self):
        if self.data is None or len(self.data) == 0:
            return
        if len(self.data) < 2 * self.min_samples_leaf:
            self.isLeaf = True
            return

        data_set = self.data.reset_index(drop=True)
        labels = self.labels

        if self.max_features * len(data_set.columns) <= 1:
            n = 1
        else:
            n = int(np.random.choice(range(1, int(self.max_features * len(data_set.columns)) + 1), 1)[0])

        random_features = np.random.choice(data_set.columns, n, replace=False)

        best_split_gini = np.inf
        best_feature = None
        best_value = None
        best_L_idxs = None
        best_R_idxs = None

        for f in random_features:
            for x in data_set[f].values:
                L_idxs = data_set.index[data_set[f] < x].tolist()
                R_idxs = data_set.index[data_set[f] >= x].tolist()

                L_labels = labels[L_idxs]
                R_labels = labels[R_idxs]

                if (len(L_labels) < self.min_samples_leaf) or (len(R_labels) < self.min_samples_leaf):
                    continue

                gi_L = self.gini_index(L_labels)
                gi_R = self.gini_index(R_labels)
                gini_index_of_split = len(L_idxs) * gi_L + len(R_idxs) * gi_R

                if gini_index_of_split < best_split_gini:
                    best_split_gini = gini_index_of_split
                    best_feature = f
                    best_value = x
                    best_L_idxs = L_idxs
                    best_R_idxs = R_idxs

        if best_feature is None:
            self.isLeaf = True
            return

        L_labels = labels[best_L_idxs]
        R_labels = labels[best_R_idxs]

        self.left = Node(
            data=data_set.iloc[best_L_idxs].reset_index(drop=True),
            labels=L_labels,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
        )
        if (self.gini_index(L_labels) == 0) or (len(L_labels) < 2 * self.min_samples_leaf):
            self.left.isLeaf = True

        self.right = Node(
            data=data_set.iloc[best_R_idxs].reset_index(drop=True),
            labels=R_labels,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
        )
        if (self.gini_index(R_labels) == 0) or (len(R_labels) < 2 * self.min_samples_leaf):
            self.right.isLeaf = True

        self.split = (best_feature, best_value)


class Tree:
    def __init__(self, root: Node, max_depth=np.inf, min_samples_leaf: int = 1):
        self.root = root
        self.levels = [[self.root]]
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.root.min_samples_leaf = self.min_samples_leaf
        self.build_tree()

    def build_tree(self):
        while np.any([(n is not None) and (not n.isLeaf) for n in self.levels[-1]]) and (len(self.levels) < self.max_depth):
            level = []
            for node in [n for n in self.levels[-1] if (n is not None) and (not n.isLeaf)]:
                node.find_best_split()
                level.append(node.left)
                level.append(node.right)
            if len(level) == 0:
                return
            self.levels.append(level)

        if len(self.levels) >= self.max_depth:
            for n in self.levels[-1]:
                if n is not None:
                    n.isLeaf = True

    def predict(self, x):
        idx = 0
        depth = len(self.levels)
        val = Counter(self.root.labels).most_common()[0][0]
        for i in range(depth):
            node = self.levels[i][idx]
            if (node is None) or (node.split is None):
                return val
            s = node.split
            if x[list(node.data.columns).index(s[0])] < s[1]:
                if i + 1 < depth and node.left in self.levels[i + 1]:
                    idx = self.levels[i + 1].index(node.left)
                    val = Counter(node.left.labels).most_common()[0][0]
            else:
                if i + 1 < depth and node.right in self.levels[i + 1]:
                    idx = self.levels[i + 1].index(node.right)
                    val = Counter(node.right.labels).most_common()[0][0]
        return val


class RandomForestClassifier_Custom:
    def __init__(self, n_estimators=10, max_features=1.0, min_samples_leaf=1, bootstrap=False, max_depth=np.inf, random_state=None):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.max_depth = max_depth
        self.bootstrap = bootstrap
        if random_state is not None:
            np.random.seed(random_state)
        self.trees = []

    def fit(self, data, labels):
        N = len(labels)
        for _ in range(self.n_estimators):
            if self.bootstrap:
                idxs = np.random.choice(range(N), N)
                data_set = data.iloc[idxs]
                label_set = labels[idxs]
            else:
                data_set = data
                label_set = labels

            root = Node(data_set, label_set)
            root.max_features = self.max_features
            t = Tree(root, self.max_depth, self.min_samples_leaf)
            self.trees.append(t)

    def predict(self, data_points):
        if len(self.trees) == 0:
            raise RuntimeError("Classifier needs to be fitted")
        if isinstance(data_points, pd.DataFrame):
            X = data_points.values
        else:
            X = np.asarray(data_points)
        preds = []
        for x in X:
            x_preds = [t.predict(x) for t in self.trees]
            preds.append(Counter(x_preds).most_common()[0][0])
        return preds

#### Reuse Codes from Random Forest Strategy.ipynb

In [7]:
def f(x):
    if x > 0.05:
        return 1
    elif x < -0.05:
        return -1
    else:
        return 0


def profit_importance(t, df, rets, random_state=123):
    rng = np.random.RandomState(random_state)
    profits = []
    for col in df.columns:
        df2 = df.copy()
        df2[col] = rng.permutation(df[col].values)
        pred = np.array(t.predict(df2))
        profits.append(float((pred * rets).sum()))
    return profits


def random_forest_profit_importance(m, df, rets):
    return pd.DataFrame({"cols": df.columns, "pi_imp": profit_importance(m, df, rets)}).sort_values("pi_imp", ascending=True)


def random_forest_feat_importance(m, df):
    importances = m.feature_importances_
    std = np.std([tree.feature_importances_ for tree in m.estimators_], axis=0)
    forest_importances = pd.Series(importances, index=df.columns)
    forest_importances = forest_importances / forest_importances.sum()
    return forest_importances.sort_values(ascending=False)


def profit_importance_scores(model, X_val: pd.DataFrame, val_rets: np.ndarray, random_state: int = 123) -> pd.DataFrame:
    base_pred = np.array(model.predict(X_val))
    base_profit = float((base_pred * val_rets).sum())
    perm_profits = profit_importance(model, X_val, val_rets, random_state=random_state)
    imp = [base_profit - p for p in perm_profits]
    return pd.DataFrame({"cols": X_val.columns, "pi_score": imp}).sort_values("pi_score", ascending=False).reset_index(drop=True)

#### Strategy Implementation

In [8]:
DROP_COLS = [
    "ticker", "date",
    "next_period_return", "spy_next_period_return",
    "rel_performance", "pred_rel_return",
    "return", "cum_ret", "spy_cum_ret",
]


def load_and_prepare(pkl_path: str) -> pd.DataFrame:
    data = pd.read_pickle(pkl_path)
    data = data.copy()
    data.replace([np.inf, -np.inf], np.nan, inplace=True)
    try:
        data = data.groupby("ticker").ffill()
    except Exception:
        data = data.groupby(level="ticker").ffill()
    data = data.fillna(0)
    data = data.copy()
    data["rel_performance"] = data["pred_rel_return"].apply(f)
    data = data.reset_index()
    data = data.set_index("date").sort_index()
    return data


def get_feature_cols(df: pd.DataFrame) -> list:
    return [c for c in df.columns if c not in DROP_COLS]


def make_quarterly_windows(df: pd.DataFrame, train_years: int = 5, gap_months: int = 3, test_months: int = 3):
    idx = df.index
    min_date = pd.to_datetime(idx.min())
    max_date = pd.to_datetime(idx.max())

    train_months = 12 * train_years
    step_months = 3

    windows = []
    i = 0
    while True:
        train_start = min_date + pd.DateOffset(months=step_months * i)
        train_end = train_start + pd.DateOffset(months=train_months)
        test_start = train_end + pd.DateOffset(months=gap_months)
        test_end = test_start + pd.DateOffset(months=test_months)
        if test_end > max_date:
            break
        windows.append((train_start, train_end, test_start, test_end))
        i += 1
    return windows


def sample_rf_params(rng: np.random.Generator) -> dict:
    min_samples_leaf = int(rng.choice(np.arange(50, 1200 + 1, 50)))
    max_depth = int(rng.choice(np.arange(5, 40 + 1, 5)))
    n_estimators = int(rng.choice(np.arange(10, 200 + 1, 10)))
    max_features = rng.choice(["sqrt", "log2"])
    return {
        "min_samples_leaf": min_samples_leaf,
        "max_depth": max_depth,
        "n_estimators": n_estimators,
        "max_features": max_features,
    }


def tune_hyperparams_profit(X_tr: pd.DataFrame, y_tr: np.ndarray, X_val: pd.DataFrame, val_rets: np.ndarray, n_iter: int = 40, random_state: int = 123) -> dict:
    rng = np.random.default_rng(random_state)
    best_profit = -np.inf
    best_params = None
    for _ in range(n_iter):
        params = sample_rf_params(rng)
        m = RandomForestClassifier(**params, random_state=random_state, n_jobs=-1)
        m.fit(X_tr, y_tr)
        preds = np.array(m.predict(X_val))
        profit = float((preds * val_rets).sum())
        if profit > best_profit:
            best_profit = profit
            best_params = params
    return best_params


def select_features_profit(X_tr: pd.DataFrame, y_tr: np.ndarray, X_val: pd.DataFrame, val_rets: np.ndarray, base_params: dict, random_state: int = 123) -> list:
    m0 = RandomForestClassifier(**base_params, random_state=random_state, n_jobs=-1)
    m0.fit(X_tr, y_tr)

    scores = profit_importance_scores(m0, X_val, val_rets, random_state=random_state)
    ranked = scores["cols"].tolist()
    p = len(ranked)

    k_grid = list(dict.fromkeys([5, 10, 15, 20, 25, 30, 40, 50, p]))
    k_grid = [k for k in k_grid if 1 <= k <= p]

    best_k = k_grid[0]
    best_profit = -np.inf
    for k in k_grid:
        feats = ranked[:k]
        m = RandomForestClassifier(**base_params, random_state=random_state, n_jobs=-1)
        m.fit(X_tr[feats], y_tr)
        preds = np.array(m.predict(X_val[feats]))
        profit = float((preds * val_rets).sum())
        if profit > best_profit:
            best_profit = profit
            best_k = k

    return ranked[:best_k]


def max_drawdown(v: pd.Series) -> float:
    v = v.dropna()
    if len(v) == 0:
        return np.nan
    dd = v / v.cummax() - 1.0
    return float(dd.min())


def perf_metrics(strategy_v: pd.Series, spy_v: pd.Series) -> pd.DataFrame:
    df = pd.concat([strategy_v.rename("strategy"), spy_v.rename("spy")], axis=1).dropna()
    if len(df) < 4:
        return pd.DataFrame([{"Sharpe": np.nan, "Alpha": np.nan, "Beta": np.nan}])

    strat_ret = df["strategy"].pct_change().dropna()
    spy_ret = df["spy"].pct_change().dropna()
    strat_ret, spy_ret = strat_ret.align(spy_ret, join="inner")

    ann = np.sqrt(4.0)  # quarterly -> annualize
    sharpe = float((strat_ret.mean() / strat_ret.std()) * ann) if strat_ret.std() != 0 else np.nan

    lr = linregress(spy_ret.values, strat_ret.values)
    beta = float(lr.slope)
    alpha_q = float(lr.intercept)
    alpha = float(alpha_q * 4.0)

    excess = strat_ret - spy_ret
    ir = float((excess.mean() / excess.std()) * ann) if excess.std() != 0 else np.nan
    vol = float(strat_ret.std() * ann)
    mdd = max_drawdown(df["strategy"])

    years = (df.index.max() - df.index.min()).days / 365.25
    cagr = float(df["strategy"].iloc[-1] ** (1 / years) - 1) if years > 0 else np.nan
    spy_cagr = float(df["spy"].iloc[-1] ** (1 / years) - 1) if years > 0 else np.nan

    return pd.DataFrame([{
        "Sharpe": sharpe,
        "Alpha": alpha,
        "Beta": beta,
        "IR": ir,
        "Vol": vol,
        "MaxDrawdown": mdd,
        "CAGR": cagr,
        "SPY_CAGR": spy_cagr,
        "R2": float(lr.rvalue ** 2),
    }])


def run_periodic_update_strategy(
    data: pd.DataFrame,
    train_years: int = 5,
    update_years: int = 5,
    val_years: int = 1,
    hp_search_iter: int = 40,
    random_state: int = 123,
):
    windows = make_quarterly_windows(data, train_years=train_years)
    if len(windows) == 0:
        raise ValueError("Not enough data to form any train/test windows.")

    base_params = {"n_estimators": 100, "max_depth": 35, "min_samples_leaf": 50, "max_features": "sqrt"}

    update_steps = int((update_years * 12) / 3)  # quarters
    cur_feats = None
    cur_params = None

    values = []
    dates = []
    v = 1.0

    for i, (train_start, train_end, test_start, test_end) in enumerate(windows):
        train_frame = data.loc[train_start:train_end].copy()
        test_frame = data.loc[test_start:test_end].copy()
        if len(train_frame) == 0 or len(test_frame) == 0:
            continue

        feat_cols = get_feature_cols(train_frame)
        if len(feat_cols) == 0:
            continue

        if (cur_feats is None) or (cur_params is None) or (i % max(update_steps, 1) == 0):
            val_cut = pd.to_datetime(train_end) - pd.DateOffset(years=val_years)
            sub_train = data.loc[train_start:val_cut].copy()
            sub_val = data.loc[val_cut:train_end].copy()
            if len(sub_train) == 0 or len(sub_val) == 0:
                sub_train = train_frame
                sub_val = train_frame.iloc[:0].copy()

            X_tr = sub_train.reset_index()[feat_cols].copy()
            y_tr = sub_train["rel_performance"].values
            X_val = sub_val.reset_index()[feat_cols].copy()
            val_rets = sub_val["next_period_return"].values

            scaler = MinMaxScaler()
            float_vars = [c for c in feat_cols if X_tr[c].dtype in [np.float64, np.float32, float, int]]
            if len(float_vars) > 0:
                X_tr[float_vars] = scaler.fit_transform(X_tr[float_vars])
                if len(X_val) > 0:
                    X_val[float_vars] = scaler.transform(X_val[float_vars])

            if len(X_val) == 0:
                cur_feats = feat_cols
                cur_params = base_params
            else:
                cur_feats = select_features_profit(X_tr, y_tr, X_val, val_rets, base_params, random_state=random_state)
                cur_params = tune_hyperparams_profit(X_tr[cur_feats], y_tr, X_val[cur_feats], val_rets, n_iter=hp_search_iter, random_state=random_state)

        X_train = train_frame.reset_index()[cur_feats].copy()
        y_train = train_frame["rel_performance"].values
        X_test = test_frame.reset_index()[cur_feats].copy()
        test_rets = test_frame["next_period_return"].values

        scaler = MinMaxScaler()
        float_vars = [c for c in cur_feats if X_train[c].dtype in [np.float64, np.float32, float, int]]
        if len(float_vars) > 0:
            X_train[float_vars] = scaler.fit_transform(X_train[float_vars])
            X_test[float_vars] = scaler.transform(X_test[float_vars])

        m = RandomForestClassifier(**cur_params, random_state=random_state, n_jobs=-1)
        m.fit(X_train, y_train)

        preds = np.array(m.predict(X_test), dtype=float)
        profit = float((preds * test_rets).sum())
        denom = float(np.sum(np.abs(preds)))
        if denom > 0:
            v = v + (v / denom) * profit

        dates.append(pd.to_datetime(test_end))
        values.append(v)

    strat_v = pd.Series(values, index=pd.to_datetime(dates), name="strategy").sort_index()
    strat_v = strat_v / strat_v.iloc[0]

    spy = data[["spy_cum_ret"]].copy()
    spy = spy[~spy.index.duplicated(keep="first")]
    spy_q = spy.resample("QE").ffill()
    spy_q = spy_q.loc[strat_v.index.min():strat_v.index.max()]
    spy_q = spy_q["spy_cum_ret"] - spy_q["spy_cum_ret"].iloc[0] + 1.0
    spy_q = spy_q.reindex(strat_v.index, method="ffill")

    metrics = perf_metrics(strat_v, spy_q)
    return strat_v, spy_q, metrics


def main():
    pkl_path = "data_small.pkl"
    data = load_and_prepare(pkl_path)

    for upd in [5, 3]:
        strat_v, spy_v, metrics = run_periodic_update_strategy(
            data, train_years=5, update_years=upd, val_years=1, hp_search_iter=40, random_state=123
        )
        print("\n" + "=" * 80)
        print(f"5-year training window | update features+HP every {upd} years")
        print(metrics.to_string(index=False))
        print("=" * 80)


if __name__ == "__main__":
    main()


5-year training window | update features+HP every 5 years
  Sharpe    Alpha     Beta        IR      Vol  MaxDrawdown     CAGR  SPY_CAGR       R2
0.702584 0.049348 0.056596 -0.246591 0.076845    -0.195411 0.054836  0.080433 0.004293

5-year training window | update features+HP every 3 years
  Sharpe    Alpha     Beta       IR      Vol  MaxDrawdown     CAGR  SPY_CAGR       R2
0.678319 0.049302 0.028932 -0.26355 0.076181    -0.307089 0.052473  0.080433 0.001142
